# Filtering parallel corpus using character-level edit distance

* Remove near-identical sentence pairs <br>
* Remove heavily paraphrased pairs <br>
* Retain pairs that most likely reflects lexical or morphosyntactic differences <br>

In [4]:
import pandas as pd
import Levenshtein

In [ ]:
def edit_ratio(s1, s2):
    # compute character-level edit distance ratio
    dist = Levenshtein.distance(s1, s2)
    # normalize by the length of the longer sentence to get a ratio
    return dist / max(len(s1), len(s2), 1)

# lower ratio indicates more similar sentence pairs
# higher ratio indicates more different sentence pairs

# ratio < 0.15 : too similar (almost identical), REMOVE
# ratio 0.15–0.65 : KEEP
# ratio > 0.65 : too different (paraphrase), REMOVE

In [ ]:
# filter parallel sentence pairs based on edit distance ratio
def filter_pairs(df, src_col, tgt_col, low=0.15, high=0.65):
    kept = [] # filtered pairs kept for training
    removed = [] # discarded pairs
    
    # iterate through each sentence pair
    for _, row in df.iterrows():
        # convert to string and remove leading/trailing whitespace
        s1 = str(row[src_col]).strip()
        s2 = str(row[tgt_col]).strip()
        
        # compute normalized edit distance ratio
        ratio = edit_ratio(s1, s2)
        
        # keep pairs within the desired range
        if low <= ratio <= high:
            kept.append((s1, s2, ratio))
        else:
            # remove pairs that are too similar or too different
            removed.append((s1, s2, ratio))
    
    # convert results back to DataFrames
    kept_df = pd.DataFrame(kept, columns=[src_col, tgt_col, "ratio"])
    removed_df = pd.DataFrame(removed, columns=[src_col, tgt_col, "ratio"])

    return kept_df, removed_df

In [ ]:
df = pd.read_csv("../data/bilingual/original_bilingual.csv")

kept_df, removed_df = filter_pairs(df, "nk", "sk")

print("Kept:", len(kept_df))
print("Removed:", len(removed_df))

kept_df.to_csv("../data/bilingual/original_bilingual_filtered.csv", index=False)

Kept: 74870
Removed: 55868
